# Qwen annotation runner
Choose a GPU runtime. Upload the supplied blinded JSON input to the runtime as annotation_input.json, then run both cells.

In [ ]:
%pip install bitsandbytes==0.50.2 accelerate==1.15.0 transformers==5.18.0


In [ ]:
from pathlib import Path
# Upload the blinded annotation_input.json file to the Colab runtime first.
PACKET_RAW = Path('/content/annotation_input.json').read_text()
def parse(raw):
    def unique(pairs):
        d={}
        for k,v in pairs:
            if k in d:raise ValueError('Duplicate key')
            d[k]=v
        return d
    a=json.loads(raw,object_pairs_hook=unique)
    assert isinstance(a,dict) and set(a)=={'label','confidence','notes'}
    assert a['label'] in ['reversible','partially_reversible','irreversible']
    assert a['confidence'] in ['low','medium','high'] and isinstance(a['notes'],str) and a['notes'].strip()
    return a


import hashlib, json, time
from pathlib import Path
from importlib.metadata import version
import torch
from google.colab import drive, files
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, GenerationConfig, set_seed

assert torch.cuda.is_available(), 'Select a GPU runtime.'
drive.mount('/content/drive')
MODEL = 'Qwen/Qwen2.5-7B-Instruct'
REV = 'a09a35458c702b33eeacc393d103063234e8bc28'
data = json.loads(PACKET_RAW)
h = hashlib.sha256(PACKET_RAW.encode()).hexdigest()
assert set(data) == {'rules', 'records'}
assert all(set(r) == {'annotation_id', 'environment', 'task', 'state', 'candidate', 'horizon'} for r in data['records'])
system = 'Label action recovery using only the supplied scenario rules. Respond with one JSON object with label, confidence, and notes. label must be reversible, partially_reversible, or irreversible. confidence must be low, medium, or high. Keep notes brief. The horizon counts calls after the candidate.'
config = {'packet_sha256': h, 'model': MODEL, 'revision': REV, 'system': system,
          'seed': 42, 'sampling': False, 'max_new_tokens': 192, 'quantization': 'NF4 double, float16 compute, SDPA',
          'versions': {k: version(k) for k in ('torch', 'transformers', 'accelerate', 'bitsandbytes')}}
set_seed(42)
tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REV)
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                           bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(MODEL, revision=REV, quantization_config=quant,
                                          dtype=torch.float16, device_map={'': 0}, attn_implementation='sdpa')
model.eval()
generation = GenerationConfig(max_new_tokens=192, do_sample=False, num_beams=1,
                              pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
                              eos_token_id=model.generation_config.eos_token_id)
directory = Path('/content/drive/MyDrive/ADNN_Irreversibility/annotations/task3_cross_environment') / h[:12]
directory.mkdir(parents=True, exist_ok=True)
out = directory / 'task3_qwen_answers.json'
answers = {}
if out.exists():
    existing = json.loads(out.read_text())
    assert existing['config'] == config
    answers = {a['annotation_id']: a for a in existing['answers']}
for row in data['records']:
    ident = row['annotation_id']
    msg = [{'role': 'system', 'content': system},
           {'role': 'user', 'content': data['rules'][row['environment']] + '\n\n' + json.dumps(row)}]
    if ident in answers:
        assert answers[ident]['messages'] == msg
        continue
    rendered = tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(rendered, return_tensors='pt', add_special_tokens=False).to('cuda')
    assert inputs.input_ids.shape[1] <= 4096
    with torch.inference_mode():
        output = model.generate(**inputs, generation_config=generation)
    raw = tokenizer.decode(output[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)
    try:
        annotation = parse(raw)
    except (ValueError, AssertionError, TypeError, KeyError):
        annotation = None
    answers[ident] = {'annotation_id': ident, 'messages': msg, 'raw_response': raw, 'annotation': annotation}
    temporary = out.with_suffix('.tmp')
    temporary.write_text(json.dumps({'config': config, 'answers': list(answers.values())}, indent=2))
    temporary.replace(out)
    print(f'{len(answers)}/{len(data["records"])} saved')
assert len(answers) == len(data['records'])
print(f"Done. Valid answers: {sum(a['annotation'] is not None for a in answers.values())}/{len(answers)}")
files.download(str(out))
